# Notebook 05: Renewable Share

**One Sensor, One Year — Edition 1: India Breathes**

The big question: how much of India's electricity came from renewables in 2024? How does that share shift day by day, season by season?

We define "clean" two ways:
- **Renewables only:** Wind + Solar + Other RE
- **All clean:** Renewables + Nuclear + Hydro (zero direct CO2)

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])

# Compute share columns
df24['re_share'] = (df24['renewables'] / df24['total'] * 100)  # Wind+Solar+Other RE
df24['clean_share'] = ((df24['nuclear'] + df24['hydro'] + df24['renewables']) / df24['total'] * 100)
df24['fossil_share'] = ((df24['coal'] + df24['lignite'] + df24['gas']) / df24['total'] * 100)
df24['wind_solar'] = df24['wind'] + df24['solar']
df24['wind_solar_share'] = (df24['wind_solar'] / df24['total'] * 100)

print(f'Renewable share (Wind+Solar+Other RE): {df24["re_share"].mean():.1f}% average')
print(f'Clean share (RE + Nuclear + Hydro):    {df24["clean_share"].mean():.1f}% average')
print(f'Wind+Solar only:                       {df24["wind_solar_share"].mean():.1f}% average')

Renewable share (Wind+Solar+Other RE): 12.0% average
Clean share (RE + Nuclear + Hydro):    23.0% average
Wind+Solar only:                       11.4% average


## 1. Daily Renewable Share — The Year at a Glance

In [2]:
fig = go.Figure()

# Raw daily RE share
fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['re_share'],
    mode='markers', marker=dict(size=4, color='#BBBBBB'),
    name='Daily RE %', hovertemplate='RE: %{y:.1f}% | %{x|%b %d}<extra></extra>',
))

# 7-day and 30-day rolling
re7 = df24['re_share'].rolling(7, center=True).mean()
re30 = df24['re_share'].rolling(30, center=True).mean()

fig.add_trace(go.Scatter(
    x=df24['date'], y=re7, mode='lines',
    line=dict(width=2, color='#27AE60'),
    name='7-day avg', hovertemplate='7d: %{y:.1f}%<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df24['date'], y=re30, mode='lines',
    line=dict(width=2.5, color='#1A7A3A', dash='dash'),
    name='30-day avg', hovertemplate='30d: %{y:.1f}%<extra></extra>',
))

# Annual mean
fig.add_hline(y=df24['re_share'].mean(), line_dash='dot', line_color='grey',
              annotation_text=f'Annual avg: {df24["re_share"].mean():.1f}%')

# Mark peak and trough
peak = df24.loc[df24['re_share'].idxmax()]
trough = df24.loc[df24['re_share'].idxmin()]
fig.add_annotation(x=peak['date'], y=peak['re_share'],
    text=f"Peak: {peak['re_share']:.1f}%<br>{peak['date'].strftime('%b %d')}",
    showarrow=True, arrowhead=2, ax=40, ay=-30)
fig.add_annotation(x=trough['date'], y=trough['re_share'],
    text=f"Low: {trough['re_share']:.1f}%<br>{trough['date'].strftime('%b %d')}",
    showarrow=True, arrowhead=2, ax=-40, ay=30)

fig.update_layout(
    title='Daily Renewable Share (Wind + Solar + Other RE) — India 2024',
    yaxis_title='Renewable Share (%)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
)
fig.show()

print(f'Peak RE day: {peak["date"].strftime("%B %d")} — {peak["re_share"]:.1f}%')
print(f'Lowest RE day: {trough["date"].strftime("%B %d")} — {trough["re_share"]:.1f}%')

Peak RE day: May 28 — 18.9%
Lowest RE day: October 14 — 7.7%


## 2. Clean Energy Share (RE + Nuclear + Hydro)

In [3]:
fig = go.Figure()

# Clean share components stacked
clean_components = ['nuclear', 'hydro', 'wind', 'solar', 'other_re']
clean_labels = ['Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']
clean_colors = ['#2EC4B6', '#1B4F72', '#85C1E9', '#F5B041', '#A569BD']

for col, label, color in zip(clean_components, clean_labels, clean_colors):
    share = (df24[col] / df24['total'] * 100).rolling(7, center=True).mean()
    fig.add_trace(go.Scatter(
        x=df24['date'], y=share,
        name=label, mode='lines',
        line=dict(width=0.5, color=color),
        fillcolor=color, stackgroup='one',
        hovertemplate=f'{label}: %{{y:.1f}}%<extra></extra>',
    ))

fig.add_hline(y=df24['clean_share'].mean(), line_dash='dot', line_color='grey',
              annotation_text=f'Clean avg: {df24["clean_share"].mean():.1f}%')

fig.update_layout(
    title='Clean Energy Share Breakdown (% of total) — India 2024 (7-day avg)',
    yaxis_title='Share of Total Generation (%)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

# Peak clean day
peak_clean = df24.loc[df24['clean_share'].idxmax()]
print(f'Best clean energy day: {peak_clean["date"].strftime("%B %d")} — {peak_clean["clean_share"]:.1f}%')
print(f'  Nuclear: {peak_clean["nuclear"]/peak_clean["total"]*100:.1f}%, Hydro: {peak_clean["hydro"]/peak_clean["total"]*100:.1f}%, Wind: {peak_clean["wind"]/peak_clean["total"]*100:.1f}%, Solar: {peak_clean["solar"]/peak_clean["total"]*100:.1f}%')

Best clean energy day: August 26 — 36.8%
  Nuclear: 3.7%, Hydro: 14.5%, Wind: 11.5%, Solar: 6.3%


## 3. Wind + Solar — The New Energy Duo

In [4]:
fig = go.Figure()

# Wind share
wind_share = (df24['wind'] / df24['total'] * 100).rolling(7, center=True).mean()
solar_share = (df24['solar'] / df24['total'] * 100).rolling(7, center=True).mean()

fig.add_trace(go.Scatter(
    x=df24['date'], y=wind_share,
    name='Wind', mode='lines',
    line=dict(width=0.5, color='#85C1E9'),
    fillcolor='#85C1E9', stackgroup='one',
    hovertemplate='Wind: %{y:.1f}%<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df24['date'], y=solar_share,
    name='Solar', mode='lines',
    line=dict(width=0.5, color='#F5B041'),
    fillcolor='#F5B041', stackgroup='one',
    hovertemplate='Solar: %{y:.1f}%<extra></extra>',
))

fig.update_layout(
    title='Wind + Solar Share of Total Generation — India 2024 (7-day avg)',
    yaxis_title='Share (%)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

print(f'Wind+Solar combined share: {df24["wind_solar_share"].mean():.1f}% average')
print(f'Peak W+S day: {df24.loc[df24["wind_solar_share"].idxmax(), "date"].strftime("%B %d")} — {df24["wind_solar_share"].max():.1f}%')
print(f'Days where W+S > 15%: {(df24["wind_solar_share"] > 15).sum()}')
print(f'Days where W+S > 20%: {(df24["wind_solar_share"] > 20).sum()}')

Wind+Solar combined share: 11.4% average
Peak W+S day: May 28 — 18.4%
Days where W+S > 15%: 31
Days where W+S > 20%: 0


## 4. Monthly Clean Energy Progression

In [5]:
df24['month'] = df24['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly_re = df24.groupby('month')['re_share'].mean()
monthly_clean = df24.groupby('month')['clean_share'].mean()
monthly_ws = df24.groupby('month')['wind_solar_share'].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=month_names, y=monthly_clean.values,
    name='All Clean (RE+Nuclear+Hydro)',
    marker_color='#27AE60',
    text=[f'{v:.0f}%' for v in monthly_clean.values],
    textposition='outside',
))
fig.add_trace(go.Bar(
    x=month_names, y=monthly_re.values,
    name='Renewables only (W+S+Other)',
    marker_color='#85C1E9',
    text=[f'{v:.0f}%' for v in monthly_re.values],
    textposition='outside',
))
fig.add_trace(go.Bar(
    x=month_names, y=monthly_ws.values,
    name='Wind+Solar only',
    marker_color='#F5B041',
    text=[f'{v:.0f}%' for v in monthly_ws.values],
    textposition='outside',
))

fig.update_layout(
    title='Monthly Clean Energy Share (%) — India 2024',
    yaxis_title='Share (%)',
    barmode='group',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

## 5. Cumulative Clean Energy Through the Year

In [6]:
# Cumulative generation: fossil vs clean
df24['cum_fossil'] = (df24['coal'] + df24['lignite'] + df24['gas']).cumsum()
df24['cum_clean'] = (df24['nuclear'] + df24['hydro'] + df24['renewables']).cumsum()
df24['cum_total'] = df24['total'].cumsum()
df24['cum_clean_pct'] = df24['cum_clean'] / df24['cum_total'] * 100

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['cum_fossil']/1000,
    name='Cumulative Fossil (TWh)', mode='lines',
    line=dict(width=2, color='#922B21'),
    fill='tozeroy', fillcolor='rgba(146,43,33,0.15)',
    hovertemplate='Fossil: %{y:.0f} TWh<extra></extra>',
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['cum_clean']/1000,
    name='Cumulative Clean (TWh)', mode='lines',
    line=dict(width=2, color='#27AE60'),
    fill='tozeroy', fillcolor='rgba(39,174,96,0.15)',
    hovertemplate='Clean: %{y:.0f} TWh<extra></extra>',
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['cum_clean_pct'],
    name='Running Clean %', mode='lines',
    line=dict(width=2, color='#2ECC71', dash='dash'),
    hovertemplate='Running clean: %{y:.1f}%<extra></extra>',
), secondary_y=True)

fig.update_layout(
    title='Cumulative Generation: Fossil vs Clean — India 2024',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.update_yaxes(title_text='Cumulative Generation (TWh)', secondary_y=False)
fig.update_yaxes(title_text='Running Clean %', secondary_y=True, showgrid=False)
fig.show()

print(f'Year-end cumulative clean share: {df24["cum_clean_pct"].iloc[-1]:.1f}%')
print(f'Total clean generation: {df24["cum_clean"].iloc[-1]/1000:.0f} TWh')
print(f'Total fossil generation: {df24["cum_fossil"].iloc[-1]/1000:.0f} TWh')

Year-end cumulative clean share: 23.1%
Total clean generation: 411 TWh
Total fossil generation: 1360 TWh


## 6. Distribution of Daily Clean Share

In [7]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df24['clean_share'], nbinsx=40,
    marker_color='#27AE60',
    name='All Clean',
    hovertemplate='%{x:.0f}%: %{y} days<extra></extra>',
))

fig.add_vline(x=df24['clean_share'].mean(), line_dash='dash', line_color='black',
              annotation_text=f'Mean: {df24["clean_share"].mean():.1f}%')
fig.add_vline(x=25, line_dash='dot', line_color='orange',
              annotation_text='25% threshold')

fig.update_layout(
    title='Distribution of Daily Clean Energy Share — India 2024',
    xaxis_title='Clean Share (%)',
    yaxis_title='Number of Days',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

print(f'Days where clean > 25%: {(df24["clean_share"] > 25).sum()} ({(df24["clean_share"] > 25).mean()*100:.0f}%)')
print(f'Days where clean > 30%: {(df24["clean_share"] > 30).sum()} ({(df24["clean_share"] > 30).mean()*100:.0f}%)')
print(f'Days where clean > 35%: {(df24["clean_share"] > 35).sum()} ({(df24["clean_share"] > 35).mean()*100:.0f}%)')

Days where clean > 25%: 112 (31%)
Days where clean > 30%: 45 (12%)
Days where clean > 35%: 3 (1%)


## 7. Save Updated Data

In [8]:
df24.to_csv('../data/processed/india_2024_derived.csv', index=False)
print(f'Saved india_2024_derived.csv ({len(df24.columns)} columns)')
print('New columns: re_share, clean_share, fossil_share, wind_solar, wind_solar_share,')
print('             cum_fossil, cum_clean, cum_total, cum_clean_pct')

Saved india_2024_derived.csv (33 columns)
New columns: re_share, clean_share, fossil_share, wind_solar, wind_solar_share,
             cum_fossil, cum_clean, cum_total, cum_clean_pct


## Key Findings

1. **Renewable share averaged ~12% in 2024** — but this masks huge seasonal swings
2. **Clean energy (RE + nuclear + hydro) averaged ~23%** — nearly 1 in 4 units of electricity
3. **The monsoon is the clean energy season:** Clean share peaks at ~35%+ in Aug-Sep, drops to ~16-18% in winter
4. **Wind+Solar are growing but still small:** ~11% combined, with wind dominant during monsoon and solar more consistent
5. **The cumulative chart shows the gap:** Fossil generation grows relentlessly; clean energy runs parallel but far below
6. **The distribution is bimodal** — winter days cluster around 16-20% clean, monsoon days around 28-35%

→ Next: Notebook 06 — Year-over-Year Comparison